In [1]:
import numpy as np
import torch
import hockey.hockey_env as h_env

from memory import ReplayBuffer
from sac import SACAgent

In [2]:
env = h_env.HockeyEnv()

ac_space = env.action_space
o_space = env.observation_space
print(ac_space)
print(o_space)
print(list(zip(env.observation_space.low, env.observation_space.high)))

Box(-1.0, 1.0, (8,), float32)
Box(-inf, inf, (18,), float32)
[(-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf)]


In [3]:
max_episodes=600
max_steps=500 
buffer = ReplayBuffer()
agent = SACAgent(env.observation_space.shape[0], env.action_space.shape[0], max_steps)

In [4]:
ob,_info = env.reset()
print(ob)
agent.actor(torch.FloatTensor(ob).unsqueeze(0))

[-3.          0.          0.          0.          0.          0.
  3.          0.          0.          0.          0.          0.
  1.09593153  0.03782558  0.          0.          0.          0.        ]


(tensor([[ 0.1352, -0.4650,  0.3330, -0.1870, -0.3780, -0.2018,  0.0887, -0.3102]],
        grad_fn=<AddmmBackward0>),
 tensor([[ 0.1604, -0.2576,  0.1588,  0.8878,  0.4746, -0.1115, -0.3907, -0.0018]],
        grad_fn=<ClampBackward1>))

In [5]:
stats = []
losses = []

In [ ]:
for i in range(max_episodes):
    # print("Starting a new episode")    
    total_reward = 0
    ob, _info = env.reset()
    done = False
    for t in range(max_steps):
        with torch.no_grad():
            a, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0))
        a = a.numpy()[0]

        (ob_new, reward, done, trunc, _info) = env.step(a)
        buffer.add((ob, a, reward, ob_new, float(done)))
        total_reward+= reward
        ob=ob_new        
        agent.update(buffer)
        if done: 
            break    
    stats.append([i,total_reward,t+1])
    # gen.reset()
    
    if ((i-1)%20==0):
        print("{}: Reward: {}".format(i, total_reward))

0: Reward: -15.827955909305592
1: Reward: -12.826387084615074
2: Reward: -29.023890399673594
3: Reward: -2.055777462348965
4: Reward: -20.292997198066654
5: Reward: -4.7514179075551715
6: Reward: -24.288101869021727
7: Reward: 0.0
8: Reward: -13.743715827583262
9: Reward: -4.9532497004013205
10: Reward: -31.72936131888885
11: Reward: 0.0
12: Reward: -38.49372546745202
13: Reward: -10.880775873808872
14: Reward: -33.241999490544224
15: Reward: -0.7311804181761251
16: Reward: -2.348499996144664
17: Reward: 0.0
18: Reward: -17.628209402432
19: Reward: -13.356366220694895
20: Reward: -37.65985559135018
21: Reward: 0.0
22: Reward: -23.628920659677203


KeyboardInterrupt: 

In [7]:
o, info = env.reset()
_ = env.render()
player2 = h_env.BasicOpponent(weak=False)

In [14]:
obs_buffer = []
reward_buffer=[]
obs, info = env.reset()
obs_agent2 = env.obs_agent_two()
for _ in range(251):
    env.render()
    with torch.no_grad():
        a1, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0))
        a1 = a1.numpy()[0]
    a2 = player2.act(obs_agent2)

    obs, r, d, t, info = env.step(np.hstack([a1,a2]))    
    obs_buffer.append(obs)
    reward_buffer.append(r)
    obs_agent2 = env.obs_agent_two()
    if d or t: 
        break
obs_buffer = np.asarray(obs_buffer)
reward_buffer = np.asarray(reward_buffer)

In [15]:
env.close()